In [ ]:
import random
import json
import re
import asyncio
import pandas as pd
import textwrap
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.utils.llm_utils import *
from vpei.utils.llm_requests_v3 import adapt_model_kwargs_for_model
from vpei.epistemic_consistency.prompts import *

In [ ]:
academic_disciplines = {
    "Natural Sciences": ["Physics", "Chemistry", "Biology", "Earth Science", "Astronomy", "Mathematics", "Environmental Science"],
    "Social Sciences": ["Psychology","Sociology","Anthropology","Political Science","Economics","Geography","Linguistics","Education"],
    "Humanities": ["Philosophy","History","Literature","Classics","Religious Studies","Art History","Musicology","Cultural Studies"],
    "Engineering & Technology": ["Mechanical Engineering","Electrical Engineering","Civil Engineering","Chemical Engineering","Computer Science","Aerospace Engineering","Materials Science","Industrial Engineering", "Robotics" ],
    "Health & Medical Sciences": ["Medicine","Nursing","Pharmacy","Public Health","Dentistry","Veterinary Medicine","Biomedical Sciences","Nutrition Science"],
    "Formal Sciences": ["Statistics", "Logic", "Systems Science", "Artificial Intelligence", "Data Science"],
    "Applied Professions": ["Law", "Business Administration", "Architecture", "Journalism & Media Studies", "Social Work"]
}
academic_fields = [field for fields in academic_disciplines.values() for field in fields]

print(len(academic_fields))  # ~50

In [ ]:
system_prompt = EXPERIMENTS['academic_abstracts']['generate_synthetic_abstracts']['system_prompt']
print(system_prompt)
print('-----------------------------------------------------------')
user_prompt_template = EXPERIMENTS['academic_abstracts']['generate_synthetic_abstracts']['user_prompt_template']
print(user_prompt_template)


In [ ]:
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name)

academic_field = "Anthropology"
user_prompt = user_prompt_template.format(academic_field=academic_field)
messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]
make_llm_request(model_name, messages, **model_kwargs)

In [ ]:
models = ["gpt-5-mini"]



n = 4 # number of abstracts to generate per academic_field
# n = 1 # for testing
async def generate_abstracts(system_prompt=system_prompt, user_prompt_template=user_prompt_template): 
    tasks = []
    for academic_discipline, academic_fields in academic_disciplines.items():
        for academic_field in academic_fields:
            for i in range(n):
                model_name = random.choice(models)
                model_kwargs = {}
                model_kwargs["reasoning_effort"] = "none"
                model_kwargs = adapt_model_kwargs_for_model(model_name, model_kwargs)
                user_prompt = user_prompt_template.format(academic_field=academic_field)
                messages = [{"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt}]
                # Instead of awaiting here, append coroutine to tasks
                tasks.append((model_name, system_prompt, user_prompt, academic_discipline, academic_field, make_llm_request_async(model_name, messages, **model_kwargs)))
    # Run all tasks concurrently
    results = await asyncio.gather(*[t[-1] for t in tasks], return_exceptions=True)
    payloads = []
    for idx, (model_name, system_prompt, user_prompt, academic_discipline, academic_field, _) in enumerate(tasks):
        response = results[idx]
        payload = {
            "model_name": model_name,
            "system_prompt": system_prompt,
            "user_prompt": user_prompt,
            "academic_discipline": academic_discipline,
            "academic_field": academic_field,
            "abstract": response,
        }
        payloads.append(payload)
    return payloads

# run the async function
payloads = await generate_abstracts(system_prompt, user_prompt_template)

df = pd.DataFrame(payloads)
df.to_csv("./data/academic_abstracts.csv", index=False)

print(f"Generated {len(df)} abstracts and saved to 'academic_abstracts.csv'.")